In [1]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm


In [12]:
### Simulations Parameter ##############
n = 1000
seed = 42
n_sim = 1000

########## Varying Parameters ##########
tau = 12
B_RF = int(n*0.7  * 2)
#########################################

X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': 0, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  0,
                'bootstrap':     True,  }


######## ONE Simulation   ########
portion_events_after_cut_train, portion_censored_after_cut_train, portion_no_events_after_cut_train, \
    wb_mse_ipcw, wb_cindex_ipcw, wb_y_pred_X_point, \
    rf_mse_ipcw, rf_y_pred_X_point, rf_std_pred_X_point, ijk_std_pred_X_point = simulation(
    seed=seed,tau=tau, data_generation_weibull_parameters=params_data_creation,params_rf=params_rf, X_pred_point=X_erwartung)

print(f'Portion of events after cut:     {round(portion_events_after_cut_train*100,2)}%,    n={round(n*0.7*portion_events_after_cut_train,0)}')
print(f'Portion of no events after cut:  {round(portion_no_events_after_cut_train*100,2)}%,    n={round(n*0.7*portion_no_events_after_cut_train,0)}')
print(f'Portion of censored after cut:   {round(portion_censored_after_cut_train*100,2)}%,   n={round(n*0.7*portion_censored_after_cut_train,0)})')

wb_y_pred_X_point[0]

Portion of events after cut:     81.0%,    n=567.0
Portion of no events after cut:  6.57%,    n=46.0
Portion of censored after cut:   12.43%,   n=87.0)


0.9497865613916734

In [2]:
### Simulations Parameter ##############
n = 1000
seed = 42
n_sim = 1000

########## Varying Parameters ##########
tau = 12
B_RF = int(n*0.7  * 2)
#########################################

X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'bootstrap':     True,  }


######## Start Simulation   ########

with ProcessPoolExecutor() as executor:
    
    ### Array to store the results
    portion_events_after_cut_train = np.zeros(n_sim)
    portion_censored_after_cut_train = np.zeros(n_sim)
    portion_no_events_after_cut_train = np.zeros(n_sim)
    wb_mse_ipcw = np.zeros(n_sim)
    wb_cindex_ipcw = np.zeros(n_sim)
    wb_y_pred_X_point = np.zeros(n_sim)
    rf_mse_ipcw = np.zeros(n_sim)
    rf_y_pred_X_point = np.zeros(n_sim)
    rf_std_pred_X_point = np.zeros(n_sim)
    ijk_var_pred_X_point = np.zeros(n_sim)

    futures = [
        executor.submit(
            simulation,
            seed=seed+i,
            tau=tau, 
            data_generation_weibull_parameters=params_data_creation,
            params_rf=params_rf, 
            X_pred_point=X_erwartung,
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
        _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
        _rf_mse_ipcw, _rf_y_pred_X_point, _rf_std_pred_X_point, _ijk_var_pred_X_point  = future.result()

        #Event-Stats Results
        portion_events_after_cut_train[i] = _portion_events_after_cut_train
        portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
        portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
        
        #Evaluation Results
        wb_mse_ipcw[i] = _wb_mse_ipcw
        wb_cindex_ipcw[i] = _wb_cindex_ipcw
        rf_mse_ipcw[i] = _rf_mse_ipcw

        #Prediction Results
        wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
        rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

        # Standard Deviation Estimates
        rf_std_pred_X_point[i] = _rf_std_pred_X_point
        ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point


Simulations: 100%|██████████| 1000/1000 [04:57<00:00,  3.36simulation/s]


In [4]:

print('Event-Stats Results:')
print(f'Portion of events after cut:     {round(np.mean(portion_events_after_cut_train)*100,2)}%,    n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}')
print(f'Portion of no events after cut:  {round(np.mean(portion_no_events_after_cut_train)*100,2)}%,    n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}')
print(f'Portion of censored after cut:   {round(np.mean(portion_censored_after_cut_train)*100,2)}%,   n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)})')
print('\n')

print('Evaluation Results:')
print(f'WB MSE IPCW: {np.mean(wb_mse_ipcw)}')
print(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw)}')
print(f'RF MSE IPCW: {np.mean(rf_mse_ipcw)}')
print('\n')

print('Prediction Results:')
print(f'WB Y_pred: {np.mean(wb_y_pred_X_point)}')
print(f'RF Y_pred: {np.mean(rf_y_pred_X_point)}')
print('\n')

print('EMP - Standard Deviations:')
print(f'RF EMP-STD: {np.std(rf_y_pred_X_point)}')
print(f'WB EMP-STD: {np.std(wb_y_pred_X_point)}')
print('\n')

print('Standard Deviation Estimates:')
print(f'RF STD - estimate: {np.mean(rf_std_pred_X_point)}')
print(f'IJK STD (for RF) - estimate: {np.sqrt(np.mean(ijk_var_pred_X_point))}')


Event-Stats Results:
Portion of events after cut:     81.96%,    n=574.0
Portion of no events after cut:  7.14%,    n=50.0
Portion of censored after cut:   10.9%,   n=76.0)


Evaluation Results:
WB MSE IPCW: 0.0681093234624032
WB C-Index IPCW: 0.6387974446080337
RF MSE IPCW: 0.06790798451091962


Prediction Results:
WB Y_pred: 0.9457712608311285
RF Y_pred: 0.9465540089051416


EMP - Standard Deviations:
RF EMP-STD: 0.013184060942491499
WB EMP-STD: 0.007207707764177278


Standard Deviation Estimates:
RF STD - estimate: 0.03429505365135466
IJK STD (for RF) - estimate: 0.013380333073068788
